# 7 · The particle filter

**Recap** ([theory](https://github.com/Rudip1/learn-state-estimation/blob/main/1_theory/07_particle_filter.md)).
A belief is a set of weighted samples. The bootstrap filter moves every particle with its own sample of the motion
model, multiplies its weight by the measurement likelihood $p(z\mid x^{[i]})$ (in the log domain), and resamples
when the effective sample size $\hat N_\text{eff} = 1/\sum_i (w^{[i]})^2$ falls below $M/2$. Resampling copies
particle $i$ about $Mw^{[i]}$ times: multinomial (roulette wheel) is the noisiest, systematic (one random offset,
$M$ equally spaced pointers) the least noisy and $O(M)$. Particles handle multimodal beliefs and global
localisation, but suffer from deprivation when the likelihood is sharp or the robot is kidnapped; random-particle
injection (augmented MCL) helps.

In [ ]:
# Installs the module when it is not available yet (e.g. on Colab). Building the C++ takes a few minutes.
try:
    import state_estimation
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-state-estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import state_estimation as se
from state_estimation import plotting

plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6, 4.5)})

## Resampling schemes

Ten particles with uneven weights, resampled 20 000 times by each scheme: all copy particle $i$ on average
$Mw^{[i]}$ times, but with very different spread.

In [ ]:
w = np.array([0.02, 0.25, 0.05, 0.08, 0.3, 0.01, 0.12, 0.07, 0.06, 0.04])
M = len(w)
rng = se.Rng(0)
fig, ax = plt.subplots(figsize=(9, 3.5))
for j, m in enumerate([se.Resampling.Multinomial, se.Resampling.Stratified, se.Resampling.Systematic, se.Resampling.Residual]):
    C = np.array([np.bincount(se.resample(w, m, rng), minlength=M) for _ in range(20000)])
    ax.bar(np.arange(M) + 0.2 * j - 0.3, C.std(0), width=0.2, label=f"{m.name}: mean |bias| {np.abs(C.mean(0) - M * w).max():.3f}")
ax.set_xlabel("particle"); ax.set_ylabel("std of the number of copies"); ax.legend(fontsize=8); plt.show()

### ✏️ Exercise 1 — systematic resampling and the effective sample size

Implement `ess_np(w)` (eq. 7.7) and `systematic_np(w, u0)` (eq. 7.10) for a given random offset `u0 ∈ [0, 1)`,
returning the ancestor indices. Use `np.cumsum` and a single sweep (or `np.searchsorted`).

In [ ]:
def ess_np(w):
    # ✏️ your code here
    raise NotImplementedError("exercise")

def systematic_np(w, u0):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
assert np.isclose(ess_np(np.ones(7)), 7) and np.isclose(ess_np([0, 0, 5]), 1)
assert np.isclose(ess_np(w), se.effective_sample_size(w))
for u0 in np.linspace(0, 0.999, 50):
    n = np.bincount(systematic_np(w, u0), minlength=M)
    assert n.sum() == M and np.all(n >= np.floor(M * w)) and np.all(n <= np.ceil(M * w))
print("✔ every particle is copied floor(Mw) or ceil(Mw) times")

## Monte Carlo localisation

Global localisation with six indistinguishable landmarks (the map of chapter 4), 4000 particles, scans every 5 steps.

In [ ]:
landmarks = np.array([[2.0, 2.0], [2.0, 6.0], [10.0, 2.0], [10.0, 6.0], [6.0, 4.0], [8.5, 6.5]])
dd, dt, k = se.DifferentialDrive(0.1, 0.5), 0.1, 1e-4
u = np.vstack([np.tile([0.5, 0.0], (60, 1)), np.tile([0.5, 0.5], (31, 1)), np.tile([0.5, 0.0], (80, 1))])
sensor = se.RangeBearingSensor(max_range=4.0, fov=2 * np.pi, sigma_range=0.1, sigma_bearing=0.05)
run = se.simulate_landmark_run(np.array([3.0, 1.0, 0.0]), u, dt, dd, k, sensor, landmarks, 5, se.Rng(2))
pf = se.ParticleFilterLocalization(4000, 1)
pf.init_uniform(0, 12, 0, 8)
err, ess, snaps = [], [], {}
for s, (dl, dr) in enumerate(run.wheel_travel):
    pf.predict_wheels(dd, dl, dr, 4 * k)
    if run.scans[s]:
        pf.update_landmarks_anonymous(run.scans[s], landmarks, sensor.R())
        ess.append(pf.effective_sample_size)
        pf.resample_if_needed(0.5)
    err.append(np.linalg.norm(pf.mean()[:2] - run.truth[s + 1, :2]))
    if s + 1 in (5, 15, 40):
        snaps[s + 1] = pf.particles
fig, axes = plt.subplots(1, 4, figsize=(17, 3.3))
for ax, (kk, P) in zip(axes, snaps.items()):
    ax.plot(landmarks[:, 0], landmarks[:, 1], "*", color="C1", ms=10, mec="k")
    ax.plot(*run.truth[kk, :2], "o", color="w", mec="k", ms=7)
    ax.plot(P[:, 0], P[:, 1], ".", ms=2, color="C4", alpha=0.5)
    ax.set_xlim(0, 12); ax.set_ylim(0, 8); ax.set_aspect("equal"); ax.set_title(f"step {kk}")
axes[3].semilogy(err); axes[3].set_xlabel("step"); axes[3].set_title("error of the weighted mean [m]")
plt.show()
print(f"final error {err[-1]:.3f} m; N_eff after each scan (first 10):", np.round(ess[:10], 0))

### ✏️ Exercise 2 — particles converge to the Kalman posterior

For a Gaussian prior on the pose and a position fix (a measurement linear in $x, y$), the exact posterior is the
Kalman update. Implement `kalman_fix(mu, P, z, R)` returning the posterior mean, then measure the error of the
particle filter's mean for $M = 100, 1000, 10\,000, 100\,000$ (average over 20 seeds). It should fall like
$1/\sqrt{M}$.

In [ ]:
def kalman_fix(mu, P, z, R):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
mu = np.array([1.0, -0.5, 0.3]); P = np.array([[0.5, 0.1, 0.02], [0.1, 0.3, -0.01], [0.02, -0.01, 0.05]])
z, R = np.array([1.4, -0.2]), np.diag([0.2, 0.1])
exact = kalman_fix(mu, P, z, R)
Ms, errs = [100, 1000, 10000, 100000], []
for M_ in Ms:
    e = []
    for seed in range(20):
        f = se.ParticleFilterLocalization(M_, seed); f.init_gaussian(mu, P); f.update_position_fix(z, R)
        e.append(np.linalg.norm(f.mean()[:2] - exact[:2]))
    errs.append(np.mean(e))
plt.loglog(Ms, errs, "o-", label="particle filter"); plt.loglog(Ms, errs[0] * np.sqrt(Ms[0] / np.array(Ms)), "k--", label="∝ 1/√M")
plt.xlabel("particles M"); plt.ylabel("error of the mean"); plt.legend(); plt.show()
assert errs[-1] < errs[0] / 10

## 🔨 Break it — a sharp sensor starves the particles

Track a known start with 100 particles and a sensor ten to twenty times more precise than before. Almost every
particle falls outside the narrow likelihood: $\hat N_\text{eff}$ collapses to one and the estimate jumps from one
survivor to the next. Inflating $R$ in the filter (×10 in standard deviation) is a deliberate approximation that
keeps more particles alive; 1000 particles help too.

In [ ]:
sharp = se.RangeBearingSensor(4.0, 2 * np.pi, 0.005, 0.002)
run_s = se.simulate_landmark_run(np.array([3.0, 1.0, 0.0]), u, dt, dd, k, sharp, landmarks, 5, se.Rng(2))
def track(M_, R_filter, sensor_run):
    f = se.ParticleFilterLocalization(M_, 2); f.init_gaussian(sensor_run.truth[0], np.diag([0.04, 0.04, 0.01]))
    e, n = [], []
    for s, (dl, dr) in enumerate(sensor_run.wheel_travel):
        f.predict_wheels(dd, dl, dr, k)
        if sensor_run.scans[s]:
            f.update_landmarks(sensor_run.scans[s], landmarks, R_filter); n.append(f.effective_sample_size)
            f.resample_if_needed(0.5)
        e.append(np.linalg.norm(f.mean()[:2] - sensor_run.truth[s + 1, :2]))
    return np.array(e), np.array(n)
for M_ in (100, 1000):
    for infl in (1, 10):
        e, n = track(M_, sharp.R() * infl**2, run_s)
        print(f"M = {M_:4d}, R × {infl**2:3d}: median error {np.median(e):.3f} m, median N_eff {np.median(n):6.1f}")

## 🔨 Break it — the kidnapped robot

At step 100 the robot is carried 4 m west. Plain MCL has no particle there and never recovers. Augmented MCL
(eq. 7.14) injects random particles when the measurements suddenly fit worse than they used to; with the injected
fraction capped at 10 % the lucky ones can multiply. The estimate is the mean of the cluster around the best particle.

In [ ]:
run_k = se.simulate_landmark_run(np.array([3.0, 1.0, 0.0]), u, dt, dd, k, sensor, landmarks, 1, se.Rng(2))
def kidnap(augmented, seed):
    f = se.ParticleFilterLocalization(3000, seed); f.init_gaussian(run_k.truth[0], np.diag([0.04, 0.04, 0.01]))
    e, inj = [], []
    for s, (dl, dr) in enumerate(run_k.wheel_travel):
        f.predict_wheels(dd, dl, dr, k)
        xt = run_k.truth[s + 1] + (np.array([-4.0, 0.0, 0.0]) if s + 1 >= 100 else 0)
        if (s + 1) % 2 == 0:
            le = f.update_landmarks(sensor.observe(xt, landmarks, se.Rng(1000 + s)), landmarks, sensor.R())
            if augmented:
                f.augmented_resample(le, 0.02, 0.3, 0, 12, 0, 8, max_fraction=0.1)
            else:
                f.resample_if_needed(0.5)
        P_, w_ = f.particles, f.weights
        best = P_[np.argmax(w_)]
        near = np.linalg.norm(P_[:, :2] - best[:2], axis=1) < 0.5
        e.append(np.linalg.norm(P_[near, :2].mean(0) - xt[:2])); inj.append(f.last_injection_fraction)
    return np.array(e), np.array(inj)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for seed, color in ((2, "C0"), (3, "C1"), (4, "C2")):
    for aug, ls in ((False, ":"), (True, "-")):
        e, inj = kidnap(aug, seed)
        axes[0].semilogy(e, ls, color=color, label=f"seed {seed}, {'augmented' if aug else 'plain'}")
        if aug:
            axes[1].plot(inj, color=color, label=f"seed {seed}")
        bad = np.nonzero(e[99:] >= 0.3)[0]
        print(f"seed {seed}, {'augmented' if aug else 'plain    '}: error below 0.3 m from step "
              f"{100 + (bad[-1] + 1 if len(bad) else 0)} on" if not len(bad) or bad[-1] + 1 < len(e) - 99 else
              f"seed {seed}, {'augmented' if aug else 'plain    '}: not recovered (final error {e[-1]:.2f} m)")
for a in axes: a.axvline(100, color="k", ls=":")
axes[0].set_ylabel("error [m]"); axes[0].legend(fontsize=7); axes[1].set_ylabel("injected fraction"); axes[1].legend(fontsize=8)
plt.show()

## What to remember

- Particles represent any belief; their error falls like $1/\sqrt{M}$ whatever the shape.
- Sample the motion model per particle, weight by the likelihood in log space, resample when $N_\text{eff}$ is low.
- Systematic resampling is unbiased, $O(M)$ and the least noisy.
- Sharp likelihoods and kidnapping starve the filter; more particles, a tempered likelihood and bounded random
  injection are the standard remedies.